# Lab 4: Statistical Inference in Linear Models

R's `airquality` dataset contains 153 daily records from New York, collected from May through September 1973. We'll use ozone concentration (`Ozone`, in parts per billion), solar radiation (`Solar.R`, in Langleys), and maximum daily temperature (`Temp`, in °F).

First, fit a simple regression of ozone on solar radiation and interpret the slope, confidence interval, and p-value. Then use simulations to study LINE conditions, Type I error, and power, and compare the slope t-test with a permutation test. Finally, add temperature and compare the overall association with the association holding temperature fixed.

In [ ]:
source("lab4_packages.R")
if (lab4_ensure_packages()) {
    library(ottr)
    source("lab4_feedback.R")
    aq <- na.omit(airquality[, c("Ozone", "Solar.R", "Temp")])
    solar_data <- data.frame(x=aq$Solar.R, y=aq$Ozone)
    cat("Complete-case days:", nrow(aq), "\n")
    print(head(aq))
}

## Part 1: Inference in Simple Linear Regression (4 points)

**Question 1.1.** Write `fit_slr(data)` for a data frame with columns `x` and `y`. Fit `y ~ x` and return a named list containing:

- `model`: the fitted `lm` object;
- `beta1`: its slope estimate;
- `se`: the slope's standard error;
- `p_value`: the two-sided p-value for a zero population slope.

Apply `fit_slr()` to `solar_data` and save the result as `solar_fit`. Your function should also work with other data frames containing `x` and `y`.

_Points:_ 2

In [ ]:
fit_slr <- function(data) {
    # YOUR CODE HERE
}

solar_fit <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_1.R")

In [ ]:
# Supplied display code
solar_interval <- confint(solar_fit$model, parm=2)
initial_summary <- data.frame(estimate=solar_fit$beta1, se=solar_fit$se,
    ci_lower=solar_interval[1], ci_upper=solar_interval[2], p_value=solar_fit$p_value)
print(initial_summary, row.names=FALSE)
plot(aq$Solar.R, aq$Ozone, xlab="Solar radiation (Langley)", ylab="Ozone (ppb)", pch=19, col="#406A8E")
abline(solar_fit$model, col="#B35B36", lwd=2)

**Question 1.2.** Calculate `ozone_change_20`, the fitted difference in mean ozone for a **20-Langley increase** in solar radiation.

Set `initial_statements` to the numbers of all correct statements:

1. The fitted slope is an estimate of a population association.
2. The fitted difference is the amount ozone changes on every individual day.
3. The confidence interval describes uncertainty about the population slope under the model's inference conditions.
4. The confidence interval describes the range containing 95% of individual ozone measurements.
5. A small p-value supports a nonzero slope under the test's conditions; study design and model adequacy still matter.

_Points:_ 1

In [ ]:
ozone_change_20 <- NULL # YOUR CODE HERE
initial_statements <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_2.R")

### When is the slope test valid?

Use the LINE conditions from Lecture #7 to answer the next question.

**Question 1.3.** Set `validity_statements` to the numbers of all correct statements:

1. Fixed, evenly spaced predictor values can be used in a simulation of the usual slope test.
2. Increasing the number of daily observations guarantees independent errors.
3. Equal error variance is compatible with a mean response that changes with the predictor.
4. The usual slope test requires the predictor values to follow a normal distribution.

Set `null_experiment` to the number of the experiment that estimates the slope test's **Type I error rate**. All three experiments hold the same nonconstant predictor values fixed and use independent normal errors with mean zero and standard deviation 2:

1. Generate $Y_i=2+0.4x_i+\varepsilon_i$ and count slope p-values below 0.05.
2. Generate $Y_i=2+\varepsilon_i$ and count slope p-values below 0.05.
3. Generate $Y_i=2+\varepsilon_i$ and count slope p-values at least 0.05.

_Points:_ 1

In [ ]:
validity_statements <- NULL # YOUR CODE HERE
null_experiment <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_3.R")

## Part 2: Model Assumptions and Permutation Testing (7 points)

In Part 1, we obtained a p-value from one dataset. Here in Part 2, you will examine LINE violations and compare the t-test with shuffling. These experiments show why the model behind a p-value matters.

`gen_normal()` draws the `x` values independently from a uniform distribution. With `ordered=TRUE`, it uses evenly spaced `x` values instead. `gen_dependent_errors()` makes errors from nearby observations tend to move together. Each error has standard deviation `sigma`. Use this function as given.

In [ ]:
gen_normal <- function(n, beta1=0, sigma=2, ordered=FALSE) {
    x <- if (ordered) seq(-5,5,length.out=n) else runif(n,-5,5)
    list(x=x,y=2+beta1*x+rnorm(n,sd=sigma))
}
gen_dependent_errors <- function(n, sigma, phi=.8) {
    eps <- numeric(n)
    eps[1] <- rnorm(1,sd=sigma)
    if (n>1L) for (i in 2:n) eps[i] <- phi*eps[i-1]+rnorm(1,sd=sigma*sqrt(1-phi^2))
    eps
}

**Question 2.1.** Write `sim_slope(gen_data, reps, alpha=0.05, ...)` to estimate the rejection rate of the two-sided slope test for $H_0:\beta_1=0$ at significance level `alpha`, using `reps` independently generated datasets. Return one numeric proportion.

The generator returns a list with `x` and `y`; pass the arguments in `...` to it. Your function must work under both null and alternative models.

_Points:_ 2

In [ ]:
sim_slope <- function(gen_data, reps, alpha = 0.05, ...) {
    # YOUR CODE HERE
}

In [ ]:
. = ottr::check("tests/q2_1.R")

The next experiment uses `n=50`, `sigma=2`, and a significance level of 0.05. Each case is compared with a model that has independent normal errors and chooses its `x` values in the same way. For dependent errors, both models use evenly spaced `x` values, and each error has standard deviation `sigma`. The other comparisons use independent uniform `x` values.

A violation won't necessarily increase the rejection rate.

**Question 2.2.** Write `gen_violation(n, sigma, condition)` to return a list containing numeric vectors `x` and `y`. Implement the four cases below. Keep the noise independent of `x` except in the unequal-variance case, and use the supplied function for dependent errors.

| `condition` | Predictor | Response |
|---|---|---|
| `"skew"` | independent Uniform(−5,5) | $2+\sigma(Z_i^2-1)/\sqrt{2}$, with independent $Z_i\sim N(0,1)$ draws |
| `"unequal"` | independent Uniform(−5,5) | $2+\varepsilon_i$, with independent normal errors having SD $\sigma[0.5+(x_i+5)/5]$ |
| `"dependent"` | `seq(-5,5,length.out=n)` | $2+$ `gen_dependent_errors(n,sigma)` |
| `"quadratic"` | independent Uniform(−5,5) | $2+0.25x_i^2+\varepsilon_i$, with independent normal errors having SD `sigma` |

The skewed errors have mean zero and standard deviation `sigma`. In the quadratic case, the symmetric predictor distribution gives a zero population linear slope even though the mean response is curved.

_Points:_ 2

In [ ]:
gen_violation <- function(n, sigma, condition) {
    # YOUR CODE HERE
}

In [ ]:
. = ottr::check("tests/q2_2.R")

In [ ]:
# Supplied experiment and plotting code
set.seed(414)
conditions <- c("skew","unequal","dependent","quadratic")
reference_rates <- c(
    random=sim_slope(gen_normal,reps=1200,n=50,sigma=2),
    ordered=sim_slope(gen_normal,reps=1200,n=50,sigma=2,ordered=TRUE))
violation_rates <- setNames(vapply(conditions,function(condition)
    sim_slope(gen_violation,reps=1200,n=50,sigma=2,condition=condition),numeric(1)),conditions)
matched_rates <- reference_rates[c("random","random","ordered","random")]
names(matched_rates) <- conditions
rate_comparison <- data.frame(condition=conditions,reference=as.numeric(matched_rates),
    changed=as.numeric(violation_rates),difference=as.numeric(violation_rates-matched_rates))
print(rate_comparison,row.names=FALSE)
barplot(rbind(reference=matched_rates,changed=violation_rates),beside=TRUE,
    col=c("#86A7C1","#B35B36"),ylab="Estimated rejection rate",ylim=c(0,1),legend.text=TRUE,
    args.legend=list(x="topleft",bty="n"))
abline(h=.05,lty=2)

**Question 2.3.** In `rate_comparison`, find the condition with the largest increase over its matched reference. Save its name in `largest_excess`; include all names if there is an exact tie.

Set `calibration_statements` to the numbers of all correct statements:

1. A comparison with the matched reference helps isolate the feature that changed.
2. The highest bar establishes which violation is most harmful in every regression problem.
3. Different implementations can produce different bar heights because the estimates contain Monte Carlo variation.
4. A rate close to 0.05 establishes that the corresponding LINE condition was satisfied.

_Points:_ 1

In [ ]:
largest_excess <- NULL # YOUR CODE HERE
calibration_statements <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_3.R")

### Shuffling the slope

The usual slope test gets its p-value from a t distribution. A permutation test offers another way to assess the slope's significance without assuming normal errors by building a null reference by shuffling the observed data. This makes it worth considering when errors are skewed and the sample is small. In the experiment below, we'll compare the two tests' false-positive rates using small datasets with independent, skewed outcomes.

Shuffling `y` changes its pairing with `x` while keeping the observed values. For this comparison, the null model has independent observations and the same outcome distribution at every predictor value. Under those conditions, no particular pairing of `x` and `y` is special, so shuffled slopes provide a reference for the observed slope.

Permutation testing avoids the normal-error assumption, but it still requires the shuffling assumptions above, so it does not resolve every LINE violation.

**Question 2.4.** Write `perm_test_slope(x,y,n_perm)` to return a two-sided permutation p-value for the regression slope. Use exactly `n_perm` independent shuffles of `y` relative to `x`, without replacement within each shuffle.

Return the raw fraction of shuffled absolute slopes at least as large as the observed absolute slope, including ties. The result must be a single number between zero and one. Allow for numerical roundoff when identifying ties.

_Points:_ 2

In [ ]:
perm_test_slope <- function(x, y, n_perm) {
    # YOUR CODE HERE
}

In [ ]:
. = ottr::check("tests/q2_4.R")

In [ ]:
# Supplied comparison: same independent skewed null, same n and error scale.
set.seed(414)
skew_test_rates <- replicate(250, {
    d <- gen_violation(n=10,sigma=2,condition="skew")
    c(t_test=summary(lm(d$y~d$x))$coefficients[2,4]<.05,
      permutation=perm_test_slope(d$x,d$y,n_perm=300)<.05)
})
print(rowMeans(skew_test_rates))

# Precomputed shuffles show the solar-data calculation without supplying its implementation.
solar_permutation_p <- perm_test_slope(solar_data$x,solar_data$y,n_perm=2000)
cat("Solar-data empirical-tail p-value:",solar_permutation_p,"\n")
null_slopes <- read.csv("slope_permutations_reference.csv")$slope
hist(null_slopes,breaks=30,col="#86A7C1",main="Shuffled solar radiation slopes",xlab="Slope (ppb per Langley)")
abline(v=c(-1,1)*abs(solar_fit$beta1),col="#B35B36",lwd=2)

Both tests use the same simulated datasets so we can compare them directly. Generating new random datasets may give slightly different rejection rates.

The same shuffling assumptions apply to the ozone data. Errors from nearby days may be related, so we should not simply assume they are independent.

## Part 3: Covariate Adjustment and Power (9 points)

We want to describe the association between solar radiation and mean ozone among days with similar temperatures. First compare the two models in `airquality`. Then use a simplified population model to study what those models detect and how sample size affects detection after adjustment.

**Question 3.1.** Write `fit_solar_models(data)` for a data frame containing `Ozone`, `Solar.R`, and `Temp`. Return a named list with two fitted models:

- `unadj`: `Ozone ~ Solar.R`;
- `adj`: `Ozone ~ Solar.R + Temp`.

Fit both models to the same rows in `aq` and save the list as `solar_models`.

_Points:_ 1.5

In [ ]:
fit_solar_models <- function(data) {
    # YOUR CODE HERE
}
solar_models <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_1.R")

In [ ]:
solar_comparison <- do.call(rbind,lapply(solar_models[c("unadj","adj")],function(model) {
    coefficient <- summary(model)$coefficients["Solar.R",]
    interval <- confint(model,"Solar.R",level=.95)
    data.frame(estimate=unname(coefficient[1]),se=unname(coefficient[2]),
        ci_lower=unname(interval[1]),ci_upper=unname(interval[2]),p_value=unname(coefficient[4]))
}))
print(solar_comparison)
local({
    oldpar <- par(no.readonly=TRUE); on.exit(par(oldpar))
    layout(matrix(1:2,ncol=1),heights=c(1.25,1))
    par(mar=c(4.2,4.5,2.6,1),mgp=c(2.7,.8,0))
    colors <- colorRampPalette(c("#8FC2DD","#D58557"))(100)
    index <- 1+floor(99*(aq$Temp-min(aq$Temp))/diff(range(aq$Temp)))
    plot(aq$Solar.R,aq$Ozone,pch=19,col=colors[index],
         xlab="Solar radiation (Langley)",ylab="Ozone (ppb)",
         main="Warmer colors: warmer days")

    par(mar=c(4.5,8.3,3.3,1),mgp=c(2.7,.8,0))
    positions <- c(2,1)
    bounds <- range(c(0,solar_comparison$ci_lower,solar_comparison$ci_upper))
    padding <- .18*diff(bounds)
    plot(NA,xlim=bounds+c(-padding,padding),ylim=c(.45,2.6),
         yaxt="n",xlab="Solar radiation slope (ppb per Langley)",ylab="",
         main="Estimated slopes and 95% confidence intervals",bty="n")
    axis(2,at=positions,labels=c("Unadjusted","Adjusted for\ntemperature"),las=1,tick=FALSE)
    abline(v=0,lty=2,col="gray45")
    text(0,2.42,"Zero slope",pos=4,cex=.75,col="gray35")
    arrows(solar_comparison$ci_lower,positions,solar_comparison$ci_upper,positions,
           angle=90,code=3,length=.07,lwd=3,col="#406A8E")
    points(solar_comparison$estimate,positions,pch=19,cex=1.35,col="#182D40")
    text(solar_comparison$estimate,positions+.20,
         labels=sprintf("Estimate: %.3f",solar_comparison$estimate),cex=.8)
    text(solar_comparison$estimate,positions-.22,
         labels=sprintf("95%% CI: %.3f to %.3f",solar_comparison$ci_lower,solar_comparison$ci_upper),cex=.8)
    mtext("Dot = estimated slope; capped line = model-based 95% confidence interval",
          side=3,line=.45,cex=.72)
})


Compare the solar radiation estimates, model-based 95% intervals, and p-values before and after adjustment.

**Question 3.2.** Calculate `adjusted_change_20`, the fitted difference in mean ozone for a 20-Langley increase in solar radiation **holding temperature fixed**.

Set `adjustment_statements` to the numbers of all correct statements:

1. The unadjusted and adjusted coefficients summarize different population associations.
2. A smaller adjusted coefficient implies that the unadjusted p-value should be reported instead.
3. Both solar radiation p-values are below 0.05 in these fits, while the estimated magnitudes and intervals differ.
4. The difference between coefficients quantifies a causal effect of temperature.

Set `temperature_rule` to the best way to choose a model when the question compares days at similar temperatures:

1. Retain temperature only when its own p-value is below 0.05.
2. Choose whether to adjust from the scientific question and plausible confounding, rather than from temperature's own p-value.
3. Choose whichever model produces the smaller solar radiation p-value.

_Points:_ 1.5

In [ ]:
adjusted_change_20 <- NULL # YOUR CODE HERE
adjustment_statements <- NULL # YOUR CODE HERE
temperature_rule <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_2.R")

### Simulating the two models

In the simulation, `x1`, `x2`, and `y` play the roles of solar radiation, temperature, and ozone, using arbitrary units. The numerical settings describe a hypothetical population. We'll compare the same two model structures, then ask how much data we need to detect the association **holding `x2` fixed**.

Generate each observation independently using

$$x_2\sim N(0,1),\quad z\sim N(0,1),\quad
x_1=\rho x_2+\sqrt{1-\rho^2}\,z,\quad
y=\beta_1x_1+\beta_2x_2+\varepsilon,\quad\varepsilon\sim N(0,\sigma^2).$$

Here, `x1` combines `x2` with independent noise `z` to give the predictors population correlation `rho`; `y` is a weighted sum of both predictors plus independent normal noise with standard deviation `sigma`.

Draw $x_2$, $z$, and $\varepsilon$ independently, with $|\rho|<1$. The correlation between the predictors is $\rho$ in the population; it will vary from sample to sample.

**Question 3.3.** Write `sim_mlr(n,beta1,beta2,rho,sigma,reps)` to estimate rejection rates for the two-sided tests of a zero `x1` coefficient in `y ~ x1` and `y ~ x1 + x2`, under the model above, at significance level 0.05.

Base both estimates on the same `reps` independent simulated datasets, each containing `n` observations. Return a named list with `unadj` and `adj` containing the respective rejection proportions. Your function should work with different values of all six arguments.

_Points:_ 2

In [ ]:
sim_mlr <- function(n, beta1, beta2, rho, sigma, reps) {
    # YOUR CODE HERE
}

In [ ]:
. = ottr::check("tests/q3_3.R")

In [ ]:
# Supplied settings for Questions 3.4 and 3.5
planning_parameters <- list(beta1=.3,beta2=1,rho=.4,sigma=1)
sample_sizes <- c(30,60,120,240)
simulation_reps <- 600

**Question 3.4.** Use the supplied `planning_parameters` for a simulated version of the same comparison: different `x1` values at fixed `x2`.

Set `coefficient_meaning` to the best interpretation of `beta1=0.3`:

1. Each individual outcome increases by exactly 0.3 whenever `x1` increases by one unit.
2. Mean `y` is 0.3 higher for a one-unit increase in `x1`, holding `x2` fixed.
3. The predictor `x1` explains 30% of the variation in `y`.

Set `power_component` to `"unadj"` or `"adj"` to choose the output from `sim_mlr()` that assesses detection of this **fixed-`x2` association**.

_Points:_ 1

In [ ]:
coefficient_meaning <- NULL # YOUR CODE HERE
power_component <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_4.R")

**Question 3.5.** Use your `sim_mlr()` function to estimate rejection rates for both models at each sample size in `sample_sizes`. Keep `planning_parameters` fixed and use `reps=simulation_reps`.

Save one value per sample size, in the same order as `sample_sizes`:

- `unadjusted_rates`: the `unadj` results.
- `adjusted_power`: the `adj` results.

Both models have nonzero population `x1` slopes in this setting, so the rates estimate power for the associations they test. Run the supplied display cell below to see the table and curves before continuing.

In [ ]:
set.seed(414)
unadjusted_rates <- NULL # YOUR CODE HERE
adjusted_power <- NULL # YOUR CODE HERE

In [ ]:
# Supplied display code
print(data.frame(sample_size=sample_sizes,unadjusted=as.numeric(unadjusted_rates),
                 adjusted=as.numeric(adjusted_power)),row.names=FALSE)
plot(sample_sizes,unadjusted_rates,type="b",pch=1,lty=2,lwd=2,col="#858585",
     ylim=c(0,1),xaxt="n",xlab="Sample size",ylab="Estimated rejection rate",
     main="Overall and adjusted associations")
lines(sample_sizes,adjusted_power,type="b",pch=19,lwd=2,col="#406A8E")
axis(1,at=sample_sizes,labels=sample_sizes)
abline(h=.8,lty=3,col="#B35B36")
legend("bottomright",legend=c("Unadjusted: overall association",
    "Adjusted: holding x2 fixed","80% target for adjusted test"),
    pch=c(1,19,NA),lty=c(2,1,3),lwd=c(2,2,1),
    col=c("#858585","#406A8E","#B35B36"),bty="n",cex=.85)

**Question 3.5 (continued).** Use the table and curves above to compare the models. Our study question compares responses at the same `x2` value, so base the sample-size choice on the adjusted results.

Set `selected_n` to the **smallest tested sample size in `sample_sizes`** whose corresponding value in **your `adjusted_power` vector** is at least 0.80. Use `NA` if none qualify.

Set `planning_statements` to the numbers of all correct statements:

1. The adjusted curve matches the question comparing responses at the same value of `x2`.
2. We should use whichever curve is higher, even when the question holds `x2` fixed.
3. New random draws can change the estimated rates and which sample size meets the target.
4. Reaching 80% power guarantees a statistically significant result in a study.

_Points:_ 2

In [ ]:
selected_n <- NULL # YOUR CODE HERE
planning_statements <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_5.R")

**Question 3.6.** Return to the ozone analysis. Use `solar_models` to set `synthesis_statements` to the numbers of all correct statements:

1. Multiplying the adjusted solar radiation coefficient and both endpoints of its 95% interval by 20 gives an estimated mean difference and interval for a 20-Langley increase at fixed temperature.
2. This interval describes the differences in ozone that 95% of individual days must exhibit.
3. Because both solar radiation p-values are below 0.05, the two models answer the same population question.
4. A small adjusted p-value does not remove the need to assess the model's inference conditions.

Set `final_claim` to the best conclusion from the ozone analysis:

1. The adjusted solar radiation slope estimates a positive association among days with similar temperatures.
2. The coefficient change estimates how much of solar radiation's effect is caused by temperature.
3. The significant unadjusted slope establishes that solar radiation causes ozone to increase.

_Points:_ 1

In [ ]:
synthesis_statements <- NULL # YOUR CODE HERE
final_claim <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_6.R")

## Congratulations, you are finished!!

To submit your assignment:

1. Select `Kernel -> Restart Kernel and Run All Cells...` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all tests passed.
3. Download your notebook using `File -> Download`, then upload your notebook to Gradescope.
4. Stick around while the Gradescope autograder grades your work. Make sure you see that all tests have passed on Gradescope.
5. Check that you have a confirmation email from Gradescope and save it as proof of your submission.